In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance

In [2]:
df = pd.read_csv("../data/ACTUAL_qol.csv")
df['age_health_interaction'] = df['age'] * df['health_new']
df['years_married'] = (df['year'] - df['agewed']).fillna(0) # years since marriage (Year - age when wed)
df.head(5)

,year,id,age,sex_new,race_new,degree_new,educ,relig,marital,agewed,divorce,widowed,wrkstat,happy_new,hapmar,health_new,life_new,age_health_interaction,years_married
0,1973,1,54,0,0,0,6,1,1,29.0,2.0,NaN,1,0,1.0,2,2,108,1944.0
1,1973,2,51,1,0,0,8,1,1,21.0,2.0,NaN,7,2,1.0,3,2,153,1952.0
2,1973,3,36,1,0,0,11,1,1,17.0,2.0,NaN,1,1,1.0,4,2,144,1956.0
3,1973,4,32,0,0,1,12,1,1,27.0,2.0,NaN,1,1,1.0,4,2,128,1946.0
4,1973,5,54,1,0,0,8,1,1,18.0,2.0,NaN,7,1,2.0,3,2,162,1955.0


In [3]:
# Define features and target
X = df.drop(columns=['happy_new', 'id']).fillna(-1)
y = df['happy_new']

# Split and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_improved = RandomForestClassifier(
    n_estimators=200, 
    class_weight='balanced',  # Fixes the class imbalance problem
    max_depth=15,              # Prevents the model from "over-memorizing" noise
    min_samples_leaf=5,        # Ensures the model generalizes better
    random_state=42, 
    n_jobs=-1
)
rf_improved.fit(X_train, y_train)

# Predict
predictions = rf_improved.predict(X_test)
predictions

array([2, 1, 2, ..., 2, 1, 2], shape=(8071,))

In [4]:
# accuracy score
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

# confusion matrix
cm = confusion_matrix(y_test, predictions)
cm

0.5957130467104448
              precision    recall  f1-score   support

           0       0.39      0.47      0.43      1133
           1       0.71      0.57      0.63      4457
           2       0.56      0.70      0.62      2481

    accuracy                           0.60      8071
   macro avg       0.55      0.58      0.56      8071
weighted avg       0.62      0.60      0.60      8071



array([[ 538,  452,  143],
       [ 712, 2525, 1220],
       [ 138,  598, 1745]])

In [5]:
# Cross-Validation --> model works across different subsets of data
cv_scores = cross_val_score(rf_improved, X_train, y_train, cv=5)
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")

# Permutation Importance --> "Explainability"
result = permutation_importance(rf_improved, X_test, y_test, n_repeats=10, random_state=42)

# results
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': result.importances_mean
}).sort_values(by='Importance', ascending=False)

print(importance_df.head(10))

Mean CV Accuracy: 0.6014
                   Feature  Importance
12                  hapmar    0.074861
14                life_new    0.035014
7                  marital    0.008264
11                 wrkstat    0.003878
13              health_new    0.002912
2                  sex_new    0.002218
3                 race_new    0.000161
16           years_married    0.000124
15  age_health_interaction    0.000087
9                  divorce   -0.000161
